# Etterforskning

Vi har en stor ferdiglaget graf, med titusenvis av noder.  Vi skal bruke Neo4J til å lete etter kriminalitet.


In [ ]:
%%bash
PWD=$(pwd)
mkdir -p neo4j
mkdir -p neo4j/data
mkdir -p neo4j/logspo
mkdir -p neo4j/plugins
mkdir -p neo4j/import

# Dette laster ned to plugins fra Neo4j (som må importeres for hånd om vi skal kjøre på VDI)

podman run \
    -p 7474:7474 -p 7687:7687 \
    --userns=keep-id \
    -e NEO4J_PLUGINS='["apoc", "graph-data-science"]' \
    -e NEO4J_dbms_security_procedures_unrestricted='gds.*,apoc.*' \
    -e NEO4J_dbms_security_procedures_allowlist='gds.*,apoc.*' \
    -e NEO4J_apoc_import_file_enabled=true \
    -v $PWD/neo4j/data:/data:Z \
    -v $PWD/neo4j/logs:/logs:Z \
    -v $PWD/neo4j/import:/import:Z \
    -v $PWD/neo4j/plugins:/plugins:Z \
    -e NEO4J_AUTH=neo4j/password \
    -d docker.io/library/neo4j:latest

Om dette feiler, forsøke igjen; databasen skal laste ned utvidelser og initalisere seg.

In [2]:
# Få Kontakt
from neo4j import GraphDatabase

URI = "bolt://localhost:7687"
AUTH = ("neo4j", "password")

driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()
print("Kontakt!")

Kontakt!


In [ ]:
# Sjekke APOC
records, summary, keys = driver.execute_query(
    """RETURN apoc.version() AS version;""")

print(f"Server Address: {summary.server.address}")
print("Keys:")
for k in range(len(keys)):
    print(f"\t{keys[k]}: {records[k]}")
#

print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

Server Address: 127.0.0.1:7687
Keys:
	version: <Record version='2025.11.2'>
Ressursbruk
	Kjøringen: 43ms
	Å konsumere: 0ms


In [4]:
# Sjekke GDS
records, summary, keys = driver.execute_query(
    """
    CALL gds.version();
    """)

print("Keys:")
for k in range(len(keys)):
    print(f"\t{keys[k]}: {records[k]}")
#

Keys:
	gdsVersion: <Record gdsVersion='2.24.0'>


Vi flytter grafen vi skal arbeide med

In [7]:
%%bash
cp grafer/Komplett.graphml neo4j/import

In [48]:
# Ikke starte med gamle data noe sted
records, summary, keys = driver.execute_query(
    """CALL gds.graph.drop('cliqueFinderGraph', false) YIELD graphName"""
)
records, summary, keys = driver.execute_query(
    """match (n) detach delete n""")
print("OK")

OK


In [49]:
# Lese inn grafen vi har bygget tidligere
records, summary, keys = driver.execute_query(
    """CALL apoc.import.graphml("Komplett.graphml", {storeNodeIds: true, readLabels: true})"""
    )
# for enkelt å pakke opp svaret
for record in records:
    record_dict = record.data()
#
for k in record_dict:
    print(f"\t{k}: {record_dict[k]}")
#
print("Grafen")
print(f"\tNye noder: {summary.counters.nodes_created}")
print(f"\tNye kanter: {summary.counters.relationships_created}")
print("Begge er 0 fordi dette bare gir mening når Cypher koden Per Se genererer noder")
      
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

	file: Komplett.graphml
	source: file
	format: graphml
	nodes: 56823
	relationships: 103537
	properties: 57619
	time: 2256
	rows: 0
	batchSize: -1
	batches: 0
	done: True
	data: None
Grafen
	Nye noder: 0
	Nye kanter: 0
Begge er 0 fordi dette bare gir mening når Cypher koden Per Se genererer noder
Ressursbruk
	Kjøringen: 1ms
	Å konsumere: 2259ms


## Lete etter et esel
Vi ser etter en klikk hvor alle kjenner hverandre.  Vi finner kanskje mange.  Det er interessant om (nesten) alle (eller i det minste flertallet) har uttak av kontanter.
Dette er typisk GDS-mat (algoritmer som kjører på hele grafen, ikke bare på enkeltnoder).

In [50]:
# Først, sjekke at det vi leter faktisk er der (sånn for sikkerhets skyld)
records, summary, keys = driver.execute_query(
  """
    MATCH (N:Person {Esel:True}) return N limit 20
  """
)
print(f"Antall i settet er {len(records)}")

Antall i settet er 16


Vi leter etter en klikk, hvor alle kjenner hverandre.  Vi starter med å hente ut alle personer.  Vi henter altså ikke ut konti og transkaskjoner.  Det er mer enn 100.000 kanter i EPOST-settet så mye "blir igjen".

In [51]:
# Hente personer og kun Kjenner-relasjonen
records, summary, keys = driver.execute_query(
  """CALL 
    gds.graph.project(
      'cliqueFinderGraph',
      'Person',              // Type node
      {
        Kjenner: {           //  Type på kanter
          type: 'Kjenner',
          orientation: 'UNDIRECTED' // Betrakter dem som om de var uten retning
        }
      }
    )
  YIELD nodeCount, relationshipCount, projectMillis;
  """
)
# Det skal ikke komme data tilbake men info om subgrafen.  Det vil si det kommer
# en dict med info om hvor mange noder og kanter som ble hentet
for record in records:
  record_dict = record.data()
  for k in record_dict:
      print(f"\t{k}: {record_dict[k]}")
    #
#

print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

	nodeCount: 56670
	relationshipCount: 730
	projectMillis: 41
Ressursbruk
	Kjøringen: 0ms
	Å konsumere: 51ms


Nå trenger vi en liste over folk i klikker, slik at vi vet hvor vi skal lete.  SOm vi så tidligere så er klikk en "sterk" datastruktur som ikke så lett oppstår tilfeldig, heller ikke i grafer uten skala.

I virkeligheten ville vi ha forsøkt en rekke verdier og undersøkt settet for hver verdi.  Vi kan imidlertid kortslutte litt

In [ ]:
# Så skal vi lete etter en klikk av folk
records, summary, keys = driver.execute_query(
    """CALL 
    gds.kcore.stream('cliqueFinderGraph')
    YIELD nodeId, coreValue
    WITH gds.util.asNode(nodeId) AS n, coreValue
    WHERE coreValue >= 13  // Verdien vi forsøker
    RETURN n.id, coreValue
    """
)
# for enkelt å pakke opp svaret
for r in records:
    print(r)
#
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

<Record n.id='9f8c2e35-ad72-4b1a-9efa-e69b8e189026' n.Esel=True coreValue=14>
<Record n.id='f713c253-6cf8-4c58-808d-7bcdbecafff3' n.Esel=True coreValue=14>
<Record n.id='9f58a8a6-828a-4320-83bd-729e2a6f505d' n.Esel=True coreValue=14>
<Record n.id='1e4dae48-bb28-4b33-a13c-0a625b4e1c7b' n.Esel=True coreValue=14>
<Record n.id='f0af76e5-7682-4c51-865b-acfd81c8d53b' n.Esel=True coreValue=14>
<Record n.id='abb55640-e4b6-4251-b639-284e50557ae7' n.Esel=True coreValue=14>
<Record n.id='bb0ccb86-3499-4c75-af53-f5d92119029f' n.Esel=True coreValue=14>
<Record n.id='b725a564-51e8-4077-8517-dd5ec7387611' n.Esel=True coreValue=14>
<Record n.id='ee9d7f48-d42b-45ac-8524-bcdd7f1ddf46' n.Esel=True coreValue=14>
<Record n.id='de1b40c1-c48f-4f66-afb0-21d7a25934ce' n.Esel=True coreValue=14>
<Record n.id='21b14323-508f-41b5-b6c6-4a7b05eb0851' n.Esel=True coreValue=14>
<Record n.id='19059119-20f1-4a76-bf14-f083bd72e212' n.Esel=True coreValue=14>
<Record n.id='3307fd02-fda1-4ab8-8b8e-14a6f601ea2c' n.Esel=True 

Vi ser vi trolig har en klikk her.  La oss merke disse nodene, og deretter skrive dem tilbake til databasen slik at vi kan vise dem frem i Bloom eller nettleseren.

In [46]:
# Så skal vi lete etter en klikk av folk
records, summary, keys = driver.execute_query(
    """CALL 
    gds.kcore.stream('cliqueFinderGraph')
    YIELD nodeId, coreValue
    WITH gds.util.asNode(nodeId) AS N, coreValue
    WHERE coreValue >= 13  // Verdien vi forsøker
    SET N.EselMistanke = 1
    """
)
# Vi ber ikke om noe i return, så dette skal være tomt
for r in records:
    print(r)
#
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

Ressursbruk
	Kjøringen: 41ms
	Å konsumere: 0ms


Vi har merket nodene i projeksjonen, skriv det vi har merket tilbake til databsen (slik at vi kan se på resultatet)

In [47]:
# Så skal vi lete etter en klikk av folk
records, summary, keys = driver.execute_query(
    """CALL 
    gds.kcore.write('cliqueFinderGraph', {
        writeProperty: 'EselMistanke'
    })
    YIELD nodePropertiesWritten, degeneracy
    """
)
# Vi ber ikke om noe i return, så dette skal være tomt
for record in records:
  record_dict = record.data()
  for k in record_dict:
      print(f"\t{k}: {record_dict[k]}")
    #
#
#
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

	nodePropertiesWritten: 56670
	degeneracy: 14
Ressursbruk
	Kjøringen: 2ms
	Å konsumere: 268ms
